In [12]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.utils import resample   # stratified resampling with replacement

GATE_FOLDERS = {
    "gap":    "results_ablation_gap",
    "margin": "results_ablation_margin",
    "max":    "results_ablation_max",
}
ROUTERS          = ["bge-large-en-v1.5", "Reason-ModernColBERT", "Qwen3-Reranker-4B"]
ALL_K            = [3, 5, 8, 10, 12, 14]   # accuracy table: every pool size
HELDOUT_K        = [10, 12, 14]            # bootstrap: extrapolated pool sizes only
NUMBER_OF_ROUNDS = 5000
SEED             = 42      # round i uses random_state = SEED + i, so results can be reproduced

print("Gates:           ", list(GATE_FOLDERS))
print("Routers:         ", ROUTERS)
print("All K:           ", ALL_K)
print("Held-out K:      ", HELDOUT_K)
print("Bootstrap rounds:", NUMBER_OF_ROUNDS)

Gates:            ['gap', 'margin', 'max']
Routers:          ['bge-large-en-v1.5', 'Reason-ModernColBERT', 'Qwen3-Reranker-4B']
All K:            [3, 5, 8, 10, 12, 14]
Held-out K:       [10, 12, 14]
Bootstrap rounds: 5000


In [3]:
def load_one_file(gate, router, k_value):
    folder = Path(GATE_FOLDERS[gate]) / f"mmlu_test_eval_service_{k_value}_300"
    for file_path in folder.glob("*.csv"):
        if router.lower() in file_path.name.lower():
            return pd.read_csv(file_path)
    print("File not found:", gate, router, k_value)


tables = {}
for gate in GATE_FOLDERS:
    for router in ROUTERS:
        for k_value in HELDOUT_K:
            tables[(gate, router, k_value)] = load_one_file(gate, router, k_value)

print("Files loaded:", len(tables), "(expected 27)")

Files loaded: 27 (expected 27)


In [4]:
check_rows = []
for router in ROUTERS:
    for k_value in HELDOUT_K:
        gap_table    = tables[("gap",    router, k_value)]
        margin_table = tables[("margin", router, k_value)]
        max_table    = tables[("max",    router, k_value)]

        check_rows.append({
            "Router": router,
            "K": k_value,
            "Questions": len(gap_table),
            "General": (gap_table["expected_service"] == "general").sum(),
            "Specialist": (gap_table["expected_service"] != "general").sum(),
            "Same order in all 3 files": gap_table["query"].equals(margin_table["query"])
                                         and gap_table["query"].equals(max_table["query"]),
        })

check_table = pd.DataFrame(check_rows)
display(check_table)

,Router,K,Questions,General,Specialist,Same order in all 3 files
0,bge-large-en-v1.5,10,1300,300,1000,True
1,bge-large-en-v1.5,12,1500,300,1200,True
2,bge-large-en-v1.5,14,1700,300,1400,True
3,Reason-ModernColBERT,10,1300,300,1000,True
4,Reason-ModernColBERT,12,1500,300,1200,True
5,Reason-ModernColBERT,14,1700,300,1400,True
6,Qwen3-Reranker-4B,10,1300,300,1000,True
7,Qwen3-Reranker-4B,12,1500,300,1200,True
8,Qwen3-Reranker-4B,14,1700,300,1400,True


In [5]:
def balanced_accuracy(correct, is_general):
    specialist_recall = correct[~is_general].mean()
    general_recall    = correct[is_general].mean()
    return (specialist_recall + general_recall) / 2


ba_rows = []
for router in ROUTERS:
    for k_value in HELDOUT_K:
        for gate in GATE_FOLDERS:
            table      = tables[(gate, router, k_value)]
            correct    = table["is_correct"].astype(int).to_numpy()   # 1 = correct, 0 = wrong
            is_general = (table["expected_service"] == "general").to_numpy()
            ba_rows.append({"Router": router, "Gate": gate, "K": k_value,
                            "BA": balanced_accuracy(correct, is_general) * 100})

ba_table = pd.DataFrame(ba_rows).pivot_table(index=["Router", "Gate"], columns="K", values="BA", sort=False)
display(ba_table.round(2))

K                               10     12     14
Router               Gate                       
bge-large-en-v1.5    gap     83.77  72.04  70.18
                     margin  77.77  68.21  64.49
                     max     88.52  84.42  83.01
Reason-ModernColBERT gap     85.32  81.92  80.73
                     margin  81.75  77.96  76.46
                     max     51.80  50.54  50.60
Qwen3-Reranker-4B    gap     85.70  82.08  81.08
                     margin  82.05  74.83  73.40
                     max     81.78  78.25  75.70

In [8]:
results = []

for router in ROUTERS:
    for k_value in HELDOUT_K:
        correct_gap = tables[("gap", router, k_value)]["is_correct"].astype(int).to_numpy()
        is_general  = (tables[("gap", router, k_value)]["expected_service"] == "general").to_numpy()
        all_rows    = np.arange(len(correct_gap))

        for other_gate in ["margin", "max"]:
            correct_other = tables[(other_gate, router, k_value)]["is_correct"].astype(int).to_numpy()

            observed = (balanced_accuracy(correct_gap, is_general)
                        - balanced_accuracy(correct_other, is_general)) * 100

            print(f"{router}, K = {k_value}, gap vs {other_gate}")

            differences = []
            for round_number in range(NUMBER_OF_ROUNDS):
                picked = resample(all_rows, replace=True, stratify=is_general, random_state=SEED + round_number)
                differences.append((balanced_accuracy(correct_gap[picked], is_general[picked])
                                    - balanced_accuracy(correct_other[picked], is_general[picked])) * 100)

                # show how round 1 was built
                if round_number == 0:
                    print(f"  round 1 picked {len(picked)} rows = "
                          f"{is_general[picked].sum()} general + {(~is_general[picked]).sum()} specialist "
                          f"({len(np.unique(picked))} different, {len(picked) - len(np.unique(picked))} repeats)")
                    print(f"  round 1 difference = {differences[-1]:.2f}")

            low, high = np.percentile(differences, [2.5, 97.5])

            print(f"  {NUMBER_OF_ROUNDS} rounds -> observed {observed:.2f}, 95% CI [{low:.2f}, {high:.2f}]\n")

            results.append({"Router": router, "Comparison": f"gap - {other_gate}", "K": k_value,
                            "Difference": observed, "CI low": low, "CI high": high})

results_table = pd.DataFrame(results).round(2)
results_table.to_csv("gate_bootstrap.csv", index=False)
display(results_table)

bge-large-en-v1.5, K = 10, gap vs margin
  round 1 picked 1300 rows = 300 general + 1000 specialist (799 different, 501 repeats)
  round 1 difference = 6.98
  5000 rounds -> observed 6.00, 95% CI [3.77, 8.17]

bge-large-en-v1.5, K = 10, gap vs max
  round 1 picked 1300 rows = 300 general + 1000 specialist (799 different, 501 repeats)
  round 1 difference = -3.92
  5000 rounds -> observed -4.75, 95% CI [-6.90, -2.68]

bge-large-en-v1.5, K = 12, gap vs margin
  round 1 picked 1500 rows = 300 general + 1200 specialist (946 different, 554 repeats)
  round 1 difference = 4.33
  5000 rounds -> observed 3.83, 95% CI [1.08, 6.54]

bge-large-en-v1.5, K = 12, gap vs max
  round 1 picked 1500 rows = 300 general + 1200 specialist (946 different, 554 repeats)
  round 1 difference = -12.25
  5000 rounds -> observed -12.38, 95% CI [-15.08, -9.71]

bge-large-en-v1.5, K = 14, gap vs margin
  round 1 picked 1700 rows = 300 general + 1400 specialist (1058 different, 642 repeats)
  round 1 difference = 4.

,Router,Comparison,K,Difference,CI low,CI high
0,bge-large-en-v1.5,gap - margin,10,6.00,3.77,8.17
1,bge-large-en-v1.5,gap - max,10,-4.75,-6.90,-2.68
2,bge-large-en-v1.5,gap - margin,12,3.83,1.08,6.54
3,bge-large-en-v1.5,gap - max,12,-12.38,-15.08,-9.71
4,bge-large-en-v1.5,gap - margin,14,5.69,3.00,8.49
5,bge-large-en-v1.5,gap - max,14,-12.83,-15.57,-10.05
6,Reason-ModernColBERT,gap - margin,10,3.57,2.12,5.02
7,Reason-ModernColBERT,gap - max,10,33.52,30.38,36.60
8,Reason-ModernColBERT,gap - margin,12,3.96,2.33,5.54
9,Reason-ModernColBERT,gap - max,12,31.37,28.33,34.50


# Acc for all

In [13]:
accuracy_rows = []

for router in ROUTERS:
    for k_value in ALL_K:
        for gate in GATE_FOLDERS:
            gate_table    = tables[(gate, router, k_value)]
            gate_accuracy = gate_table["is_correct"].mean() * 100
            accuracy_rows.append({"Router": router, "Gate": gate, "K": k_value, "Accuracy": gate_accuracy})

accuracy_table = pd.DataFrame(accuracy_rows).pivot_table(index=["Router", "Gate"], columns="K",
                                                          values="Accuracy", sort=False).round(2)
display(accuracy_table)

K                               3      5      8      10     12     14
Router               Gate                                            
bge-large-en-v1.5    gap     91.67  89.00  82.82  80.77  71.27  69.00
                     margin  91.33  83.12  76.09  74.77  67.13  64.59
                     max     95.50  93.50  87.27  84.85  78.87  75.47
Reason-ModernColBERT gap     91.67  89.88  83.55  80.46  74.87  72.35
                     margin  91.67  88.12  81.00  75.69  69.53  66.41
                     max     57.00  44.00  38.36  35.54  31.47  30.06
Qwen3-Reranker-4B    gap     97.50  95.12  87.64  87.15  82.73  81.35
                     margin  97.50  93.25  85.09  83.69  77.13  75.82
                     max     91.50  88.25  80.18  74.85  68.40  63.00